<a href="https://colab.research.google.com/github/gorbunovr98-glitch/data-analysis/blob/main/%D0%94_%D0%97%C2%AB%D0%92%D1%80%D0%B5%D0%BC%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5_%D1%80%D1%8F%D0%B4%D1%8B%2C_%D0%BE%D0%B1%D1%89%D0%B0%D1%8F_%D1%82%D0%B5%D0%BE%D1%80%D0%B8%D1%8F%C2%BB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

In [2]:
# Список файлов (путь нужно подставить свой)
files = [
    "monthly-sales-of-company-x-jan-6.csv",
    "monthly-boston-armed-robberies-j.csv",
    "international-airline-passengers.csv",
    "mean-monthly-air-temperature-deg.csv",
    "weekly-closings-of-the-dowjones-.csv",
    "daily-total-female-births-in-cal.csv"
]

def make_stationary(series, freq=None):
    """
    series: pd.Series с индексом по времени
    freq: частота (например, 'M' для месячных, 'D' для дневных) — опционально
    Возвращает преобразованный ряд и краткое описание применённого метода
    """
    # Сначала лог, если значения положительные и есть рост дисперсии
    if (series > 0).all():
        s = np.log(series)
        method = "log"
    else:
        s = series.copy()
        method = "none"

    # Обычный дифференциал (1-й порядок)
    s_diff = s.diff().dropna()
    method += "+diff1"

    # Если есть сезонность, можно добавить сезонный дифференциал.
    # Для месячных данных сезон = 12, для дневных с недельной сезонностью = 7 и т.д.
    if freq == "M":
        seasonal_period = 12
        s_diff = s_diff.diff(seasonal_period).dropna()
        method += f"+seasonal_diff({seasonal_period})"
    elif freq == "D":
        # Часто у дневных данных недельная сезонность
        seasonal_period = 7
        s_diff = s_diff.diff(seasonal_period).dropna()
        method += f"+seasonal_diff({seasonal_period})"

    return s_diff, method

def test_stationarity(series):
    result = adfuller(series.dropna())
    p_value = result[1]
    is_stationary = p_value < 0.05
    return is_stationary, p_value

results = {}

for fname in files:
    # Предполагаем, что в CSV есть колонка с данными и колонка с датой
    df = pd.read_csv(fname)

    # ВАЖНО: здесь нужно адаптировать имена колонок под реальные в файлах
    # Например, дата может быть в колонке 'date', 'month', 'time' и т.п.
    date_col = None
    value_col = None

    # Простой автоподбор: ищем колонку с датами и колонку с числами
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]) or col.lower() in ["date", "time", "month", "year"]:
            date_col = col
        elif pd.api.types.is_numeric_dtype(df[col]):
            value_col = col

    if date_col is None or value_col is None:
        print(f"Не удалось автоматически определить колонки в {fname}. Проверьте вручную.")
        continue

    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    series = df[value_col]

    # Определение частоты (грубо по разнице между соседними датами)
    diffs = np.diff(df.index.astype(np.int64))
    if len(diffs) > 0:
        median_diff_ns = np.median(diffs)
        median_days = median_diff_ns / (24 * 3600 * 1e9)
        if abs(median_days - 30) < 5:
            freq = "M"
        elif abs(median_days - 7) < 2:
            freq = "W"
        elif abs(median_days - 1) < 0.5:
            freq = "D"
        else:
            freq = None
    else:
        freq = None

    stationary_series, method = make_stationary(series, freq)
    is_stat, pval = test_stationarity(stationary_series)

    results[fname] = {
        "original_mean": series.mean(),
        "original_std": series.std(),
        "stationary_mean": stationary_series.mean(),
        "stationary_std": stationary_series.std(),
        "method": method,
        "is_stationary": is_stat,
        "p_value": pval
    }

    print(f"{fname}: метод={method}, стационарный={is_stat}, p-value={pval:.4f}")

# Результаты можно сохранить в DataFrame для удобства
results_df = pd.DataFrame(results).T
print(results_df[["method", "is_stationary", "p_value"]])

monthly-sales-of-company-x-jan-6.csv: метод=log+diff1+seasonal_diff(12), стационарный=False, p-value=0.1668
monthly-boston-armed-robberies-j.csv: метод=log+diff1+seasonal_diff(12), стационарный=True, p-value=0.0006
international-airline-passengers.csv: метод=log+diff1+seasonal_diff(12), стационарный=True, p-value=0.0002
mean-monthly-air-temperature-deg.csv: метод=log+diff1+seasonal_diff(12), стационарный=True, p-value=0.0000
Не удалось автоматически определить колонки в weekly-closings-of-the-dowjones-.csv. Проверьте вручную.
daily-total-female-births-in-cal.csv: метод=log+diff1+seasonal_diff(7), стационарный=True, p-value=0.0000
                                                           method  \
monthly-sales-of-company-x-jan-6.csv  log+diff1+seasonal_diff(12)   
monthly-boston-armed-robberies-j.csv  log+diff1+seasonal_diff(12)   
international-airline-passengers.csv  log+diff1+seasonal_diff(12)   
mean-monthly-air-temperature-deg.csv  log+diff1+seasonal_diff(12)   
daily-total-femal